# 03 — Cross-Calibration

Calibrate Ultra460, Ultra321, and Pico017 against the Picarro (CH4) and certified gas standards (C3H8) using the Feb 12 calibration sequence.
Validate on Feb 6.  C2H6 is cross-calibrated across all WYO co-deployment days using Ultra460 as the reference.


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import re as _re
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.graph_objects as go
from scipy import stats
from pathlib import Path

plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})


In [ ]:
# Get tank and calibration period details
def parse_tank_details(filepath):
    """Parse tank_details.txt into tank concentrations and calibration windows.

    Lines starting with '#' are ignored — comment out any window line to exclude it.

    All species concentrations are stored in ppm (ppb values in the file are converted).

    Returns
    -------
    tank : dict
        {label: {CH4_ppm, C3H8_ppm, C2H6_ppm}}  — N2_zero always present; None where not certified.
    windows_by_date : dict
        {'YYYYMMDD': [window dicts with label, tank_key, stable_start, stable_end]}
    """
    LABEL_MAP = {
        'n2 zero':   'N2_zero', 'n2_zero':  'N2_zero',
        'noaa tank': 'NOAA',    'noaa':     'NOAA',
    }
    for _i in range(1, 6):
        LABEL_MAP[f'dilution{_i}']  = f'Dilution{_i}'
        LABEL_MAP[f'dilution {_i}'] = f'Dilution{_i}'

    def _norm(s):
        sl = s.strip().lower()
        if sl in LABEL_MAP:
            return LABEL_MAP[sl]
        m = _re.match(r'dilution\s*(\d)', sl)
        if m:
            return f'Dilution{m.group(1)}'
        return s.strip().replace(' ', '_')

    tank = {'N2_zero': {'CH4_ppm': 0.0, 'C3H8_ppm': 0.0, 'C2H6_ppm': 0.0}}
    windows_by_date = {}

    conc_re = _re.compile(r'^([\w\s]+):\s*(.*)')
    val_re  = _re.compile(r'([\d.]+)\s*(ppm|ppb)\s+(\w+)', _re.I)
    date_re = _re.compile(r'(\w+)\s+(\d+),\s+(\d{4})')
    win_re  = _re.compile(r'(\d{2}:\d{2}:\d{2})\s+to\s+(\d{2}:\d{2}:\d{2})\s*[-–]\s*(.+)')
    MONTHS  = {m: i + 1 for i, m in enumerate([
        'january', 'february', 'march', 'april', 'may', 'june',
        'july', 'august', 'september', 'october', 'november', 'december'])}

    current_date = None

    with open(filepath) as fh:
        for line in fh:
            line = line.rstrip()
            if not line or line.strip().startswith('#'):
                continue

            # Date header line  e.g. "February 3, 2026 (UTC times)"
            dm = date_re.search(line)
            if dm:
                month = MONTHS.get(dm.group(1).lower())
                day, year = int(dm.group(2)), int(dm.group(3))
                current_date = f'{year}{month:02d}{day:02d}'
                windows_by_date[current_date] = []
                continue

            # Calibration window line  e.g. "19:02:00 to 19:08:00 - N2 zero"
            wm = win_re.match(line.strip())
            if wm and current_date:
                t0, t1, raw_label = wm.group(1), wm.group(2), wm.group(3)
                tank_key  = _norm(raw_label)
                date_part = f'{current_date[:4]}-{current_date[4:6]}-{current_date[6:]}'
                windows_by_date[current_date].append({
                    'label':        tank_key,
                    'tank_key':     tank_key,
                    'stable_start': f'{date_part} {t0}',
                    'stable_end':   f'{date_part} {t1}',
                })
                continue

            # Tank concentration line  e.g. "NOAA: 2.0012ppm CH4, 1.63ppb C2H6"
            # Only parse before any date section
            if current_date is None:
                cm = conc_re.match(line.strip())
                if cm:
                    key   = _norm(cm.group(1))
                    entry = {'CH4_ppm': None, 'C3H8_ppm': None, 'C2H6_ppm': None}
                    for vm in val_re.finditer(cm.group(2)):
                        val  = float(vm.group(1))
                        unit = vm.group(2).lower()
                        gas  = vm.group(3).upper()
                        if gas == 'CH4':
                            entry['CH4_ppm'] = val if unit == 'ppm' else val / 1000
                        elif gas == 'C3H8':
                            entry['C3H8_ppm'] = val if unit == 'ppm' else val / 1000
                        elif gas == 'C2H6':
                            entry['C2H6_ppm'] = val / 1000 if unit == 'ppb' else val
                    tank[key] = entry

    return tank, windows_by_date


In [ ]:
def load_merged(date_str):
    df = pd.read_csv(MERGED_DIR / f'{date_str}.csv')
    df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'])
    return df


def extract_window_stats(df, windows):
    """Per-window median, mean, and std for every instrument column."""
    all_cols = [c for c in CH4_COLS + C3H8_COLS + C2H6_COLS if c in df.columns]
    rows = []
    for w in windows:
        t0  = pd.Timestamp(w['stable_start'])
        t1  = pd.Timestamp(w['stable_end'])
        sub = df[(df['TIMESTAMP'] >= t0) & (df['TIMESTAMP'] <= t1)]
        row = {'label': w['label'], 'tank_key': w['tank_key'], 'n_rows': len(sub)}
        for col in all_cols:
            row[col]           = sub[col].median()
            row[col + '_mean'] = sub[col].mean()
            row[col + '_std']  = sub[col].std()
        rows.append(row)
    return pd.DataFrame(rows)


def build_master_cal_df(windows_by_date, tank, dfs=None):
    """Merge window stats from all calibration dates into one DataFrame.

    Dates with no active windows (all commented out) are skipped.
    dfs : optional pre-loaded {date_str: DataFrame}; falls back to load_merged.
    """
    frames = []
    for date_str, windows in windows_by_date.items():
        if not windows:
            continue
        df = dfs[date_str] if dfs else load_merged(date_str)
        s  = extract_window_stats(df, windows)
        s['cal_date']      = date_str
        s['tank_CH4_ppm']  = s['tank_key'].map(lambda k: tank.get(k, {}).get('CH4_ppm'))
        s['tank_C3H8_ppm'] = s['tank_key'].map(lambda k: tank.get(k, {}).get('C3H8_ppm'))
        s['tank_C2H6_ppm'] = s['tank_key'].map(lambda k: tank.get(k, {}).get('C2H6_ppm'))
        frames.append(s)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def linreg(x, y):
    """OLS dropping NaN pairs.  Returns (slope, intercept, r2)."""
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        return np.nan, np.nan, np.nan
    sl, ic, r, *_ = stats.linregress(x[mask], y[mask])
    return sl, ic, r**2


def _rgba(hex_color, alpha):
    """Convert '#rrggbb' to 'rgba(r,g,b,alpha)'."""
    r = int(hex_color[1:3], 16)
    g = int(hex_color[3:5], 16)
    b = int(hex_color[5:7], 16)
    return f'rgba({r},{g},{b},{alpha})'


def apply_cal(series, coefs):
    """Apply linear correction: cal = (raw − intercept) / slope."""
    if coefs is None:
        return pd.Series(np.nan, index=series.index)
    return (series - coefs['intercept']) / coefs['slope']


def plot_cal_timeseries(date_str, show_types=('raw',), cal_ref_date='20260212'):
    """Interactive Plotly calibration timeseries.

    Parameters
    ----------
    date_str     : date key from CAL_DATES (e.g. '20260212')
    show_types   : any subset of ('raw', 'zero_only', 'cal_self', 'cal_ref')
                     raw       — uncorrected instrument readings
                     zero_only — subtract N2_zero offset only (slope=1, no gain correction)
                     cal_self  — corrected with this date's own low-range cal
                     cal_ref   — corrected with cal_ref_date's coefs (reveals drift)
    cal_ref_date : reference date used for 'cal_ref' type
    """
    PS = PLOT_STYLE  # local alias

    df            = CAL_DFS[date_str].copy()
    windows       = windows_by_date[date_str]
    date_label    = pd.Timestamp(date_str).strftime('%b %-d, %Y')
    ch4_inst_cols = SPECIES['CH4']['inst_cols']

    # Add on-the-fly columns for types that aren't pre-computed
    if 'zero_only' in show_types:
        zero_coefs = ZERO_COEFS['CH4'].get(date_str, {})
        for inst, col in ch4_inst_cols.items():
            if col in df.columns:
                df[col + '_zero_cal'] = apply_cal(df[col], zero_coefs.get(inst))

    if 'cal_ref' in show_types:
        ref_coefs = LOW_COEFS['CH4'].get(cal_ref_date, {})
        for inst, col in ch4_inst_cols.items():
            if col in df.columns:
                df[col + '_cal_ref'] = apply_cal(df[col], ref_coefs.get(inst))

    # Window-only slices with NaN spacers between windows
    windows_sorted = sorted(windows, key=lambda w: pd.Timestamp(w['stable_start']))
    pieces = []
    for w in windows_sorted:
        ws    = pd.Timestamp(w['stable_start'])
        we    = pd.Timestamp(w['stable_end'])
        piece = df[(df['TIMESTAMP'] >= ws) & (df['TIMESTAMP'] <= we)].copy()
        if len(piece):
            pieces.append(piece)
            spacer = pd.DataFrame([[np.nan] * len(piece.columns)], columns=piece.columns)
            spacer['TIMESTAMP'] = we + pd.Timedelta('1s')
            pieces.append(spacer)
    sub = pd.concat(pieces, ignore_index=True) if pieces else df.iloc[:0].copy()

    ref_label  = pd.Timestamp(cal_ref_date).strftime('%b %-d')
    self_label = pd.Timestamp(date_str).strftime('%b %-d')
    TYPE_STYLE = {
        'raw':       {'suffix': '',           'dash': 'solid',   'label': 'raw'},
        'zero_only': {'suffix': '_zero_cal',  'dash': 'dashdot', 'label': 'zero-only'},
        'cal_self':  {'suffix': '_low_cal',   'dash': 'dash',    'label': f'cal ({self_label})'},
        'cal_ref':   {'suffix': '_cal_ref',   'dash': 'dot',     'label': f'cal-ref ({ref_label})'},
    }

    fig = go.Figure()

    # ── CH4 traces ────────────────────────────────────────────────────────
    for inst, base_col, color in [
        ('Picarro',  'CH4_ppm_Picarro',  COLORS['Picarro']),
        ('Ultra460', 'CH4_ppm_Ultra460', COLORS['Ultra460']),
        ('Ultra321', 'CH4_ppm_Ultra321', COLORS['Ultra321']),
        ('Pico017',  'CH4_ppm_Pico017',  COLORS['Pico017']),
    ]:
        for type_key in show_types:
            sty = TYPE_STYLE[type_key]
            col = base_col + sty['suffix']
            if col not in sub.columns:
                continue
            name = f'{inst} [{sty["label"]}]'
            if inst == 'Picarro':
                pdata = sub[sub[col].notna()]
                fig.add_trace(go.Scatter(
                    x=pdata['TIMESTAMP'], y=pdata[col],
                    mode='markers',
                    marker=dict(color=color, size=PS['ts_picarro_size']),
                    yaxis='y', name=name,
                    legendgroup='CH4  (ppm)', legendgrouptitle=dict(text='CH4  (ppm)'),
                    hovertemplate='%{y:.4f}<extra>' + name + '</extra>',
                ))
            else:
                fig.add_trace(go.Scatter(
                    x=sub['TIMESTAMP'], y=sub[col],
                    mode='lines',
                    line=dict(color=color, width=PS['ts_line_width'], dash=sty['dash']),
                    yaxis='y', name=name,
                    legendgroup='CH4  (ppm)', legendgrouptitle=dict(text='CH4  (ppm)'),
                    hovertemplate='%{y:.4f}<extra>' + name + '</extra>',
                ))

    # ── C2H6 traces (ppm) ─────────────────────────────────────────────────
    for inst, col, color in [
        ('Ultra460', 'C2H6_ppm_Ultra460', COLORS['Ultra460']),
        ('Pico017',  'C2H6_ppm_Pico017',  COLORS['Pico017']),
        ('Ultra321', 'C2H6_ppm_Ultra321',  COLORS['Ultra321']),
    ]:
        if col not in sub.columns:
            continue
        fig.add_trace(go.Scatter(
            x=sub['TIMESTAMP'], y=sub[col],
            mode='lines',
            line=dict(color=color, width=PS['ts_line_width'], dash='dot'),
            yaxis='y2', name=inst,
            legendgroup='C2H6 (ppm)', legendgrouptitle=dict(text='C2H6 (ppm)'),
            hovertemplate='%{y:.6f}<extra>' + f'{inst} C2H6' + '</extra>',
        ))

    if 'C3H8_ppm_Ultra321' in sub.columns:
        fig.add_trace(go.Scatter(
            x=sub['TIMESTAMP'], y=sub['C3H8_ppm_Ultra321'],
            mode='lines',
            line=dict(color=COLORS['Ultra321'], width=PS['ts_line_width'], dash='dash'),
            yaxis='y3', name='Ultra321',
            legendgroup='C3H8 (ppm)', legendgrouptitle=dict(text='C3H8 (ppm)'),
            hovertemplate='%{y:.4f}<extra>Ultra321 C3H8</extra>',
        ))

    # ── Window shading + labels ───────────────────────────────────────────
    for w in windows:
        ws = pd.Timestamp(w['stable_start'])
        we = pd.Timestamp(w['stable_end'])
        fig.add_vrect(x0=ws, x1=we, fillcolor=W_COLORS.get(w['tank_key'], '#dddddd'),
                      opacity=0.30, layer='below', line_width=0)
        fig.add_annotation(x=ws + (we - ws) / 2, y=1.01, yref='paper',
                           text=w['label'], showarrow=False,
                           font=dict(size=PS['ts_window_font']), textangle=-55, xanchor='left')

    type_label = ' + '.join(TYPE_STYLE[t]['label'] for t in show_types if t in TYPE_STYLE)
    fig.update_layout(
        title=dict(text=f'{date_label} — CH4 calibration check  [{type_label}]', font=dict(size=13)),
        xaxis=dict(title='Time (UTC)', domain=[0.0, 0.85]),
        yaxis=dict(title='CH4 (ppm)', side='left', showgrid=True,
                   gridcolor=PS['grid_color']),
        yaxis2=dict(title='C2H6 (ppm)', side='right', overlaying='y',
                    anchor='x', showgrid=False),
        yaxis3=dict(title='C3H8 (ppm)', side='right', overlaying='y',
                    anchor='free', position=1.0, showgrid=False),
        legend=dict(font=dict(size=PS['legend_font_size']),
                    tracegroupgap=PS['legend_group_gap'],
                    groupclick='toggleitem', x=1.08, y=1, xanchor='left'),
        hovermode='x unified',
        template='plotly_white',
        width=PS['ts_fig_width'], height=PS['ts_fig_height'],
        margin=dict(r=PS['ts_margin_right']),
    )
    fig.show()


def plot_cal_scatter(species='CH4', show_type='raw', show_dates=None, show_insts=None, cal_ref_date='20260212'):
    """Calibration scatter: certified concentration vs instrument window mean ± std.

    Draws one overall regression line per instrument (toggleable in legend) and prints
    a per-date + overall fit table (slope, intercept, R², n) below the figure.

    Error bars are drawn at low opacity (PLOT_STYLE['sc_bar_alpha']).

    When correction coefs are unavailable for a date, the raw reading is shown instead
    and the hover tooltip notes the fallback.

    Parameters
    ----------
    species      : one of 'CH4', 'C2H6', 'C3H8'
    show_type    : one of 'raw', 'zero_only', 'cal_self', 'cal_ref'
    show_dates   : list of date strings to show (None = all CAL_DATES)
    show_insts   : list of instruments to show (None = all for this species)
    cal_ref_date : reference date used when show_type='cal_ref'
    """
    PS        = PLOT_STYLE  # local alias
    sp_config = SPECIES[species]
    inst_cols = sp_config['inst_cols']
    tank_col  = sp_config['tank_col']
    unit      = sp_config['unit']

    dates = list(show_dates) if show_dates is not None else list(CAL_DATES)
    insts = list(show_insts) if show_insts is not None else list(inst_cols.keys())

    DATE_SYMBOLS = {
        '20260203': 'circle',
        '20260206': 'square',
        '20260212': 'diamond',
    }
    TYPE_LABEL = {
        'raw':       'raw',
        'zero_only': 'zero-only',
        'cal_self':  'cal (self)',
        'cal_ref':   f'cal-ref ({pd.Timestamp(cal_ref_date).strftime("%b %-d")})',
    }

    # ── Axis range ────────────────────────────────────────────────────────
    valid_x = master_cal_df[master_cal_df[tank_col].notna()][tank_col]
    x_max   = float(valid_x.max()) * 1.05 if len(valid_x) else 10.0
    x_pad   = x_max * 0.04

    # ── Collect plotting data per (inst, date) ────────────────────────────
    plot_data = {}

    for inst in insts:
        col  = inst_cols[inst]
        mcol = col + '_mean'
        scol = col + '_std'

        for date_str in dates:
            if date_str not in windows_by_date:
                continue

            if show_type == 'zero_only':
                c = ZERO_COEFS[species].get(date_str, {}).get(inst)
            elif show_type == 'cal_self':
                c = LOW_COEFS[species].get(date_str, {}).get(inst)
            elif show_type == 'cal_ref':
                c = LOW_COEFS[species].get(cal_ref_date, {}).get(inst)
            else:
                c = None

            grp = master_cal_df[master_cal_df['cal_date'] == date_str]
            x_vals, y_vals, y_err, hover_texts = [], [], [], []

            for _, row in grp.iterrows():
                if pd.isna(row.get(tank_col)):
                    continue
                if mcol not in row.index or pd.isna(row[mcol]):
                    continue

                cert     = float(row[tank_col])
                mean_raw = float(row[mcol])
                std_raw  = float(row.get(scol, np.nan))

                if c is not None:
                    y        = (mean_raw - c['intercept']) / c['slope']
                    err      = std_raw / abs(c['slope'])
                    cal_note = ''
                else:
                    y        = mean_raw
                    err      = std_raw
                    cal_note = '<br><i>no cal coefs — showing raw</i>' if show_type != 'raw' else ''

                x_vals.append(cert)
                y_vals.append(y)
                y_err.append(err if np.isfinite(err) else 0.0)
                hover_texts.append(
                    f'Tank: {row["tank_key"]}<br>'
                    f'Certified: {cert:.6g} {unit}<br>'
                    f'Mean: {y:.6g} {unit}<br>'
                    f'Std: {err:.6g} {unit}'
                    + cal_note
                )

            if x_vals:
                plot_data[(inst, date_str)] = {
                    'x': x_vals, 'y': y_vals, 'err': y_err, 'hover': hover_texts,
                }

    fig = go.Figure()

    # ── Scatter traces ────────────────────────────────────────────────────
    for inst in insts:
        color     = COLORS[inst]
        bar_color = _rgba(color, PS['sc_bar_alpha'])
        for date_str in dates:
            d = plot_data.get((inst, date_str))
            if d is None:
                continue
            symbol           = DATE_SYMBOLS.get(date_str, 'circle')
            date_label_short = pd.Timestamp(date_str).strftime('%b %-d')
            fig.add_trace(go.Scatter(
                x=d['x'], y=d['y'],
                error_y=dict(type='data', array=d['err'], visible=True,
                             color=bar_color,
                             thickness=PS['sc_bar_thickness'],
                             width=PS['sc_bar_cap_width']),
                mode='markers',
                marker=dict(color=color, symbol=symbol,
                            size=PS['sc_marker_size'],
                            opacity=PS['sc_marker_alpha'],
                            line=dict(width=PS['sc_marker_border_w'],
                                      color=PS['sc_marker_border_c'])),
                name=f'{inst} — {date_label_short}',
                legendgroup=inst,
                legendgrouptitle=dict(text=inst),
                hovertemplate='%{customdata}<extra></extra>',
                customdata=d['hover'],
            ))

    # ── Overall regression line per instrument + collect fit stats ────────
    fit_rows = []
    for inst in insts:
        inst_x, inst_y = [], []

        for date_str in dates:
            d = plot_data.get((inst, date_str))
            if d is None:
                continue
            x = np.array(d['x'], dtype=float)
            y = np.array(d['y'], dtype=float)
            inst_x.extend(x)
            inst_y.extend(y)
            sl, ic, r2 = linreg(x, y)
            fit_rows.append({
                'Instrument': inst,
                'Date':       pd.Timestamp(date_str).strftime('%b %-d'),
                'slope': sl, 'intercept': ic, 'R²': r2, 'n': len(x),
            })

        xa, ya = np.array(inst_x, dtype=float), np.array(inst_y, dtype=float)
        sl, ic, r2 = linreg(xa, ya)
        fit_rows.append({'Instrument': inst, 'Date': 'Overall',
                         'slope': sl, 'intercept': ic, 'R²': r2, 'n': len(inst_x)})

        if np.isfinite(sl) and len(inst_x) >= 2:
            x_line = np.array([-x_pad, x_max])
            y_line = sl * x_line + ic
            ic_str = f'{ic:+.4f}'
            fig.add_trace(go.Scatter(
                x=x_line, y=y_line,
                mode='lines',
                line=dict(color=COLORS[inst], width=PS['sc_fit_line_width']),
                name=inst,
                legendgroup='Fit lines',
                legendgrouptitle=dict(text='Fit lines'),
                showlegend=True,
                hovertemplate=f'{inst} fit: y = {sl:.4f}x {ic_str}<extra></extra>',
            ))

    # ── 1:1 reference line ─────────────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x=[-x_pad, x_max], y=[-x_pad, x_max],
        mode='lines',
        line=dict(color=PS['sc_ref_line_color'],
                  dash=PS['sc_ref_line_dash'],
                  width=PS['sc_ref_line_width']),
        name='1:1', showlegend=True,
        legendgroup='Reference', legendgrouptitle=dict(text='Reference'),
        hoverinfo='skip',
    ))

    # ── Dummy traces: date → symbol legend group ───────────────────────────
    for date_str in dates:
        symbol          = DATE_SYMBOLS.get(date_str, 'circle')
        date_label_long = pd.Timestamp(date_str).strftime('%b %-d, %Y')
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='markers',
            marker=dict(color='gray', symbol=symbol,
                        size=PS['sc_legend_sym_size'],
                        opacity=PS['sc_marker_alpha'],
                        line=dict(width=PS['sc_marker_border_w'],
                                  color=PS['sc_marker_border_c'])),
            name=date_label_long,
            legendgroup='Date (shape)',
            legendgrouptitle=dict(text='Date (shape)'),
            showlegend=True, hoverinfo='skip',
        ))

    type_label = TYPE_LABEL.get(show_type, show_type)
    fig.update_layout(
        title=dict(text=f'{species} Calibration Scatter — {type_label}', font=dict(size=13)),
        xaxis=dict(title=f'Certified {species} ({unit})', showgrid=True,
                   gridcolor=PS['grid_color'], range=[-x_pad, x_max]),
        yaxis=dict(title=f'Instrument {species} ({unit})', showgrid=True,
                   gridcolor=PS['grid_color'], range=[-x_pad, x_max]),
        legend=dict(font=dict(size=PS['legend_font_size']),
                    tracegroupgap=PS['legend_group_gap'],
                    groupclick='toggleitem', x=1.02, y=1, xanchor='left'),
        hovermode='closest',
        template='plotly_white',
        width=PS['sc_fig_width'], height=PS['sc_fig_height'],
    )
    fig.show()

    # ── Regression summary table ───────────────────────────────────────────
    print(f'\nLinear regression [{type_label}]:  certified {species} ({unit}) (x)  →  instrument reading (y)')
    print(f'  ideal: slope=1.000, intercept=0.000\n')
    print(f'{"Instrument":12s}  {"Date":8s}  {"slope":>9s}  {"intercept":>10s}  {"R²":>8s}  {"n":>4s}')
    print('─' * 62)
    prev_inst = None
    for row in fit_rows:
        if row['Instrument'] != prev_inst and prev_inst is not None:
            print()
        prev_inst = row['Instrument']
        sl = f'{row["slope"]:.5f}'     if np.isfinite(row['slope'])     else '—'
        ic = f'{row["intercept"]:.5f}' if np.isfinite(row['intercept']) else '—'
        r2 = f'{row["R²"]:.5f}'        if np.isfinite(row['R²'])        else '—'
        bold = '  ◀' if row['Date'] == 'Overall' else ''
        print(f'{row["Instrument"]:12s}  {row["Date"]:8s}  {sl:>9s}  {ic:>10s}  {r2:>8s}  {row["n"]:>4d}{bold}')


In [ ]:
MERGED_DIR        = Path('/uufs/chpc.utah.edu/common/home/lin-group24/agm/Mobile_SLV/Data/2026/merged')
TANK_DETAILS_FILE = Path('/uufs/chpc.utah.edu/common/home/u0890904/LAIR_1/Projects/Mobile_SLV/Data/tank_details.txt')

# Parse — TAML and all windows derived from the file
TANK, windows_by_date = parse_tank_details(TANK_DETAILS_FILE)

# Named aliases for downstream cells that reference specific dates
FEB3_WINDOWS  = windows_by_date.get('20260203', [])
FEB6_WINDOWS  = windows_by_date.get('20260206', [])
FEB12_WINDOWS = windows_by_date.get('20260212', [])

print(f'Certified gas standards ({len(TANK)}):')
for k, v in TANK.items():
    print(f'  {k}: {v}')
print(f'\nCalibration dates ({len(windows_by_date)}):')
for date, wins in windows_by_date.items():
    print(f'  {date}: {[w["label"] for w in wins]}')

In [ ]:
COLORS = {
    'Picarro':  '#1f77b4',
    'Ultra460': '#ff7f0e',
    'Ultra321': '#2ca02c',
    'Pico017':  '#d62728',
}

W_COLORS = {
    'N2_zero':   '#a8d8ea',
    'NOAA':      '#b8f0b8',
    'Dilution1': '#fffacd',
    'Dilution2': '#ffd700',
    'Dilution3': '#ffa040',
    'Dilution4': '#ff6030',
    'Dilution5': '#cc44cc',
}

# Columns present in the raw merged CSVs — used by extract_window_stats
# C2H6: Ultra321 native CSV col is ppm; Ultra460/Pico017 native cols are ppb and
# are converted to ppm in the load loop (cell below) before they appear here.
CH4_COLS  = ['CH4_ppm_Picarro', 'CH4_ppm_Ultra460', 'CH4_ppm_Ultra321', 'CH4_ppm_Pico017']
C3H8_COLS = ['C3H8_ppm_Ultra321']
C2H6_COLS = ['C2H6_ppm_Ultra460', 'C2H6_ppm_Pico017', 'C2H6_ppm_Ultra321']  # all ppm

# ── Species configuration ─────────────────────────────────────────────────
# All concentrations in ppm throughout.
SPECIES = {
    'CH4': {
        'unit':      'ppm',
        'tank_col':  'tank_CH4_ppm',
        'inst_cols': {
            'Picarro':  'CH4_ppm_Picarro',
            'Ultra460': 'CH4_ppm_Ultra460',
            'Ultra321': 'CH4_ppm_Ultra321',
            'Pico017':  'CH4_ppm_Pico017',
        },
        'cal_tanks': ['N2_zero', 'NOAA', 'Dilution1'],
    },
    'C2H6': {
        'unit':      'ppm',
        'tank_col':  'tank_C2H6_ppm',
        'inst_cols': {
            'Ultra460': 'C2H6_ppm_Ultra460',
            'Pico017':  'C2H6_ppm_Pico017',
            'Ultra321': 'C2H6_ppm_Ultra321',
        },
        # Only N2_zero (0 ppm) and NOAA (0.00163 ppm) are certified — 2-pt fit only
        'cal_tanks': ['N2_zero', 'NOAA'],
    },
    'C3H8': {
        'unit':      'ppm',
        'tank_col':  'tank_C3H8_ppm',
        'inst_cols': {'Ultra321': 'C3H8_ppm_Ultra321'},
        'cal_tanks': ['N2_zero', 'Dilution1', 'Dilution2', 'Dilution3', 'Dilution4', 'Dilution5'],
    },
}

# ── Plot style — edit here to restyle all plots ───────────────────────────
PLOT_STYLE = {
    # Shared
    'grid_color':         '#eeeeee',
    'legend_font_size':   9,
    'legend_group_gap':   12,

    # Calibration timeseries
    'ts_line_width':      1.2,
    'ts_picarro_size':    4,
    'ts_window_font':     8,
    'ts_fig_width':       1150,
    'ts_fig_height':      530,
    'ts_margin_right':    160,

    # Calibration scatter
    'sc_marker_size':     10,
    'sc_marker_alpha':    0.8,
    'sc_marker_border_w': 0,
    'sc_marker_border_c': 'rgba(80,80,80,0.5)',
    'sc_bar_alpha':       0.35,
    'sc_bar_thickness':   1.5,
    'sc_bar_cap_width':   6,
    'sc_fit_line_width':  1,
    'sc_ref_line_color':  'gray',
    'sc_ref_line_dash':   'dash',
    'sc_ref_line_width':  1,
    'sc_legend_sym_size': 10,
    'sc_fig_width':       1200,
    'sc_fig_height':      580,
}


In [ ]:
# ── Load all calibration data upfront ────────────────────────────────────
# CAL_DFS   : {date_str: full merged DataFrame}  — all C2H6 in ppm
# CAL_STATS : {date_str: per-window stats DataFrame}
# master_cal_df : all dates concatenated, with certified tank concentrations joined

# Only include dates that have at least one active (un-commented) window
CAL_DATES = sorted(d for d, wins in windows_by_date.items() if wins)

CAL_DFS = {}
for _date in windows_by_date:  # load all dates (for timeseries even if excluded from scatter)
    _df = load_merged(_date)
    # Ultra460 and Pico017 log C2H6 in ppb — convert to ppm for consistency
    for _ppb_col in ['C2H6_ppb_Ultra460', 'C2H6_ppb_Pico017']:
        if _ppb_col in _df.columns:
            _df[_ppb_col.replace('_ppb_', '_ppm_')] = _df[_ppb_col] / 1000.0
    # Ultra321 already logs C2H6 in ppm natively — no conversion needed
    CAL_DFS[_date] = _df

CAL_STATS = {
    date: extract_window_stats(CAL_DFS[date], windows_by_date[date])
    for date in CAL_DATES
}

master_cal_df = build_master_cal_df(windows_by_date, TANK, dfs=CAL_DFS)

print(f'Loaded raw data for {len(CAL_DFS)} dates; {len(CAL_DATES)} have active cal windows:')
for date in CAL_DATES:
    label  = pd.Timestamp(date).strftime('%b %-d, %Y')
    n_wins = len(windows_by_date[date])
    n_rows = len(CAL_DFS[date])
    print(f'  {label}  ({date}): {n_wins} windows, {n_rows:,} merged rows')
excluded = [d for d in windows_by_date if d not in CAL_DATES]
if excluded:
    for d in excluded:
        print(f'  {pd.Timestamp(d).strftime("%b %-d, %Y")}  ({d}): all windows commented out — excluded from cal')
print(f'\nmaster_cal_df: {len(master_cal_df)} total window-stat rows across active dates')


In [ ]:
master_cal_df

In [ ]:
# ── Generic calibration coefficient computation ───────────────────────────

def compute_low_cal(master_cal_df, sp_config):
    """Multi-point linear calibration per date per instrument for a species.

    Uses sp_config['cal_tanks'] as the calibration anchor points.
    Duplicate tank_key windows are averaged before fitting.

    Returns {date_str: {inst: {'slope', 'intercept', 'r2', 'n_tanks'} or None}}
    """
    tank_col  = sp_config['tank_col']
    inst_cols = sp_config['inst_cols']
    cal_tanks = sp_config['cal_tanks']

    coefs = {}
    for date_str, grp in master_cal_df.groupby('cal_date'):
        coefs[date_str] = {}
        sub = grp[grp['tank_key'].isin(cal_tanks)]
        for inst, col in inst_cols.items():
            mcol = col + '_mean'
            if mcol not in sub.columns or sub.empty:
                coefs[date_str][inst] = None
                continue
            pts = (sub.groupby('tank_key', as_index=False)
                      .agg(x=(tank_col, 'first'), y=(mcol, 'mean')))
            xv   = pts['x'].values.astype(float)
            yv   = pts['y'].values.astype(float)
            mask = np.isfinite(xv) & np.isfinite(yv)
            n    = mask.sum()
            if n >= 2:
                sl, ic, r2 = linreg(xv[mask], yv[mask])
                coefs[date_str][inst] = {'slope': sl, 'intercept': ic, 'r2': r2, 'n_tanks': n}
            else:
                coefs[date_str][inst] = None
    return coefs


def compute_zero_cal(master_cal_df, sp_config):
    """Zero-offset-only cal: slope=1, intercept=N2_zero window mean reading.

    Multiple N2_zero windows are averaged.
    Returns {date_str: {inst: {'slope', 'intercept'} or None}}
    """
    inst_cols = sp_config['inst_cols']

    coefs = {}
    for date_str, grp in master_cal_df.groupby('cal_date'):
        coefs[date_str] = {}
        zero_rows = grp[grp['tank_key'] == 'N2_zero']
        for inst, col in inst_cols.items():
            mcol = col + '_mean'
            if mcol not in zero_rows.columns or zero_rows.empty:
                coefs[date_str][inst] = None
                continue
            zero_mean = float(zero_rows[mcol].mean())
            if np.isfinite(zero_mean):
                coefs[date_str][inst] = {'slope': 1.0, 'intercept': zero_mean}
            else:
                coefs[date_str][inst] = None
    return coefs


# ── Compute coefs for all species ─────────────────────────────────────────
LOW_COEFS  = {sp: compute_low_cal(master_cal_df, SPECIES[sp])  for sp in SPECIES}
ZERO_COEFS = {sp: compute_zero_cal(master_cal_df, SPECIES[sp]) for sp in SPECIES}

# ── Apply calibrated columns to every CAL_DFS entry ──────────────────────
for _date, _df in CAL_DFS.items():
    for _sp, _sp_config in SPECIES.items():
        for _inst, _col in _sp_config['inst_cols'].items():
            if _col not in _df.columns:
                continue
            _df[_col + '_low_cal']  = apply_cal(_df[_col], LOW_COEFS[_sp].get(_date, {}).get(_inst))
            _df[_col + '_zero_cal'] = apply_cal(_df[_col], ZERO_COEFS[_sp].get(_date, {}).get(_inst))

# ── Summary tables per species ────────────────────────────────────────────
for sp, sp_config in SPECIES.items():
    cal_tanks = sp_config['cal_tanks']
    unit      = sp_config['unit']
    inst_cols = sp_config['inst_cols']
    print(f'── {sp} ({unit})  |  cal tanks: {", ".join(cal_tanks)}')
    print(f'{"Instrument":12s}  {"slope":>9s}  {"intercept":>10s}  {"R²":>8s}  {"n_pts":>5s}')
    for date_str in CAL_DATES:
        print(f'\n  {pd.Timestamp(date_str).strftime("%b %-d, %Y")}:')
        for inst in inst_cols:
            c = LOW_COEFS[sp].get(date_str, {}).get(inst)
            if c:
                print(f'  {inst:12s}  {c["slope"]:9.5f}  {c["intercept"]:10.5f}  {c["r2"]:8.5f}  {c["n_tanks"]:5d}')
            else:
                print(f'  {inst:12s}  — no data')
    print(f'\n  Zero offsets (N2_zero window mean):')
    print(f'  {"":12s}  ' + '  '.join(f'{pd.Timestamp(d).strftime("%b %-d"):>10s}' for d in CAL_DATES))
    for inst in inst_cols:
        vals = [
            f'{ZERO_COEFS[sp].get(d, {}).get(inst, {}).get("intercept", float("nan")):10.4f}'
            if ZERO_COEFS[sp].get(d, {}).get(inst) else f'{"—":>10s}'
            for d in CAL_DATES
        ]
        print(f'  {inst:12s}  ' + '  '.join(vals))
    print()


In [ ]:
# ── Configuration — change these to explore ──────────────────────────────
DATE_STR   = '20260212'           # '20260203' | '20260206' | '20260212'
SHOW_TYPES = ['raw', 'zero_only']  # 'raw' | 'zero_only' | 'cal_self' | 'cal_ref'
CAL_REF    = '20260212'           # reference date used when 'cal_ref' is in SHOW_TYPES
# ─────────────────────────────────────────────────────────────────────────

plot_cal_timeseries(DATE_STR, show_types=SHOW_TYPES, cal_ref_date=CAL_REF)

In [ ]:
# ── Configuration — change these to explore ──────────────────────────────
SCATTER_SPECIES = 'C3H8'          # 'CH4' | 'C2H6' | 'C3H8'
SCATTER_TYPE    = 'zero_only'    # 'raw' | 'zero_only' | 'cal_self' | 'cal_ref'
SCATTER_DATES   = None           # None = all dates, or e.g. ['20260203', '20260212']
SCATTER_INSTS   = None           # None = all instruments, or e.g. ['Picarro', 'Ultra460']
CAL_REF         = '20260212'     # reference date for 'cal_ref' type
# ─────────────────────────────────────────────────────────────────────────

plot_cal_scatter(species=SCATTER_SPECIES, show_type=SCATTER_TYPE, show_dates=SCATTER_DATES,
                 show_insts=SCATTER_INSTS, cal_ref_date=CAL_REF)


In [ ]:
# ── C3H8 residual vs certified CH4 ───────────────────────────────────────
# y = Ultra321 measured C3H8 − certified tank C3H8  (Δ, should be ~0 if cal is good)
# x = certified CH4  (proxy for dilution level / concentration regime)
# Points where tank has no certified C3H8 (e.g. NOAA) are excluded.

_DATE_COLORS  = {'20260203': '#1f77b4', '20260206': '#ff7f0e', '20260212': '#2ca02c'}
_DATE_SYMBOLS = {'20260203': 'circle',  '20260206': 'square',  '20260212': 'diamond'}
_MCOL         = 'C3H8_ppm_Ultra321_mean'

_fig = go.Figure()

for _date in CAL_DATES:
    _grp = master_cal_df[master_cal_df['cal_date'] == _date].copy()
    _grp = _grp[_grp['tank_CH4_ppm'].notna() & _grp['tank_C3H8_ppm'].notna()]
    if _grp.empty or _MCOL not in _grp.columns:
        continue
    _grp['_delta'] = _grp[_MCOL] - _grp['tank_C3H8_ppm']
    _dlabel = pd.Timestamp(_date).strftime('%b %-d, %Y')
    _hover  = [
        f'Tank: {r.tank_key}<br>'
        f'CH4 cert: {r.tank_CH4_ppm:.3f} ppm<br>'
        f'C3H8 cert: {r.tank_C3H8_ppm:.4f} ppm<br>'
        f'C3H8 meas: {r[_MCOL]:.4f} ppm<br>'
        f'Δ: {r._delta:+.4f} ppm'
        for _, r in _grp.iterrows()
    ]
    _fig.add_trace(go.Scatter(
        x=_grp['tank_CH4_ppm'], y=_grp['_delta'],
        mode='markers',
        marker=dict(color=_DATE_COLORS[_date], symbol=_DATE_SYMBOLS[_date],
                    size=10, opacity=0.85,
                    line=dict(width=0.5, color='rgba(50,50,50,0.4)')),
        name=_dlabel,
        hovertemplate='%{customdata}<extra></extra>',
        customdata=_hover,
    ))

_fig.add_hline(y=0, line_dash='dash', line_color='gray', line_width=1)
_fig.update_layout(
    title=dict(text='C3H8 residual: measured − certified  vs  certified CH4', font=dict(size=13)),
    xaxis=dict(title='Certified CH4 (ppm)', showgrid=True, gridcolor='#eeeeee'),
    yaxis=dict(title='ΔC3H8  (meas − cert)  [ppm]', showgrid=True, gridcolor='#eeeeee'),
    legend=dict(font=dict(size=10)),
    hovermode='closest',
    template='plotly_white',
    width=900, height=480,
)
_fig.show()
